In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
# Summarisation middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

#1. Message based summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),   # when length of the message is greater than 10 then sumamrise the model.
            keep=("messages",4) #keep the latest/recent 4 chat.
        )
    ]
)

In [5]:
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

print(config)

{'configurable': {'thread_id': 'test-1'}}


In [6]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='b99388ca-ed6d-4623-8401-4d5588e34bf3'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks a simple math question. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 78, 'total_tokens': 110, 'completion_time': 0.065916989, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.002966677, 'prompt_tokens_details': None, 'queue_time': 0.30806434, 'total_time': 0.068883666}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04d7a-2430-7c70-a371-2aefd0aca261-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 32, 'total_tokens': 110, 'output_token_details': {'reasoning': 13}})]}
Messages: 2
Messag

In [10]:
#2. Token based summarisation

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi"""

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [11]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~737 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='0e29ddfa-9b85-4e55-b9ab-665e40243796'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function search_hotels that takes city string. We should call it.', 'tool_calls': [{'id': 'fc_a90eabb4-acd4-41bc-afff-bf1efa36b755', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 129, 'total_tokens': 184, 'completion_time': 0.117123601, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.005526915, 'prompt_tokens_details': None, 'queue_time': 0.378843412, 'total_time': 0.122650516}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_ru

In [3]:
# 3. based on fraction


from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi"""

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

# Test
cities = ["Paris", "London", "Tokyo"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context

    print(
        f"{city}: ~{tokens} tokens ({fraction:.4%}), "
        f"{len(response['messages'])} msgs"
    )
    print(response["messages"])

Paris: ~371 tokens (0.2898%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='2306db0e-d0a2-4280-9285-f7b1dd55e0cb'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. We can use the search_hotels function. Provide city "Paris".', 'tool_calls': [{'id': 'fc_b309cc9f-de1c-493e-954d-3ed4e21660bc', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 128, 'total_tokens': 178, 'completion_time': 0.104871343, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005049744, 'prompt_tokens_details': None, 'queue_time': 0.379297942, 'total_time': 0.109921087}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04d88-a4a6-75d1-

In [4]:
# Human in the loop middleware
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


In [ ]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

# whenever send_email_tool is called first human will either approve, edit or reject it. and there is no restriction on read_email_tool


In [7]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [8]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f19ff86b-73fd-414d-b33b-4c41591b3fad'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_6875377a-30fc-4f15-b16b-54b2ba5f4944', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.128224079, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007868045, 'prompt_tokens_details': None, 'queue_time': 0.378586773, 'total_time': 0.136092124}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [ ]:
# see an __interrupt__ is happening is happend to take permission from human.

In [9]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸ Paused! Approving...
✅ Result: The email has been sent to **john@test.com** with the subject **“Hello”** and the body:

> How are you?


In [10]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f19ff86b-73fd-414d-b33b-4c41591b3fad'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_6875377a-30fc-4f15-b16b-54b2ba5f4944', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.128224079, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007868045, 'prompt_tokens_details': None, 'queue_time': 0.378586773, 'total_time': 0.136092124}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [11]:
# To reject the email

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = ChatGroq(
    model="openai/gpt-oss-120b",
)
agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [12]:
vconfig = {"configurable": {"thread_id": "test-reject"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [13]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸ Paused! Approving...
✅ Result: I’m ready to send that email for you. Just to confirm, you’d like to send an email to **john@test.com** with the subject **“Hello”** and the body **“How are you?”**. Shall I go ahead and send it?


In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f72b58e1-fb30-4677-93a8-5fd6ef3147a2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_e8663551-dfbb-4094-9297-5a47c45b9af0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.128318288, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.048994708, 'prompt_tokens_details': None, 'queue_time': 0.319267719, 'total_time': 0.177312996}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4b2f03d631', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [22]:
# To edit
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [23]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to wrong@email.com with subject 'Test' and body 'Hello'"
            )
        ]
    },
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='7a235c03-a141-4beb-ae77-3edffafa1e69'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide recipient, subject, body. Use function.', 'tool_calls': [{'id': 'fc_711a762f-848e-4426-b7cc-79d37cda67dd', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 172, 'total_tokens': 237, 'completion_time': 0.135545837, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.006869503, 'prompt_tokens_details': None, 'queue_time': 0.379406813, 'total_time': 0.14241534}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls',

In [25]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                       # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

⏸ Paused! Editing...


In [26]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='7a235c03-a141-4beb-ae77-3edffafa1e69'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide recipient, subject, body. Use function.', 'tool_calls': [{'id': 'fc_711a762f-848e-4426-b7cc-79d37cda67dd', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 172, 'total_tokens': 237, 'completion_time': 0.135545837, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.006869503, 'prompt_tokens_details': None, 'queue_time': 0.379406813, 'total_time': 0.14241534}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls',